In [ ]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers trl pyyaml -q
print("Deps installed")


In [ ]:
# Cell 2: Clone CogMem + load tasks
REPO_BRANCH = "master"
!if [ ! -d /notebooks/CogMem/.git ]; then git clone --branch {REPO_BRANCH} --single-branch https://github.com/tungooxx/CogMem.git /notebooks/CogMem; else cd /notebooks/CogMem && git fetch origin && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}; fi
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
import json
from pathlib import Path
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
TASK_LIMIT = 834

if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "canonical_solution": item.get("canonical_solution", ""),
            "entry_point": item.get("entry_point", ""),
            "libs": item.get("libs", []),
        })
    with open(TASKS_PATH, "w") as f:
        for task in tasks:
            f.write(json.dumps(task) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

if TASK_LIMIT:
    tasks = tasks[:TASK_LIMIT]

print(Path('/notebooks/CogMem').resolve())
!cd /notebooks/CogMem && git branch --show-current && git rev-parse --short HEAD
print("Tasks loaded:", len(tasks))


In [ ]:
# Cell 3: Prepare split manifest + new-architecture configs
from cogmem.config import CogMemConfig
from cogmem.consolidation.experiment import (
    NewArchitectureExperimentConfig,
    prepare_new_arch_task_split,
    load_new_arch_runtime,
    run_new_arch_episode_collection,
    build_new_arch_skill_cards,
    compare_new_arch_routes,
    persist_route_memory_utility,
    run_new_arch_qstar_cycle,
)

RESET_COLLECTION_PROGRESS = False
RUN_QSTAR_TRAINING = True

NOTEBOOK_CONFIG = NewArchitectureExperimentConfig(
    experiment_dir="/notebooks/cogmem_new_architecture",
    manifest_path="/notebooks/cogmem_new_architecture/bigcodebench_manifest.json",
    memory_bank_path="/notebooks/cogmem_new_architecture/memory_bank.json",
    skills_path="/notebooks/cogmem_new_architecture/skill_cards.json",
    model_name="Qwen/Qwen2.5-3B-Instruct",
    task_limit=TASK_LIMIT,
    max_attempts=3,
    temperature=0.0,
)

split_result = prepare_new_arch_task_split(tasks, config=NOTEBOOK_CONFIG)
manifest = split_result["manifest"]
train_tasks = split_result["train_tasks"]
dev_tasks = split_result["dev_tasks"]
test_tasks = split_result["test_tasks"]

COGMEM_CONFIG = CogMemConfig(
    project_dir="/notebooks/CogMem",
    bigcodebench_memory_bank=NOTEBOOK_CONFIG.memory_bank_path,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    adapters_dir="/notebooks/cogmem_new_architecture/adapters",
    adapter_registry_path="/notebooks/cogmem_new_architecture/adapters/registry.json",
    experiments_dir="/notebooks/cogmem_new_architecture/experiments",
    logs_dir="/notebooks/cogmem_new_architecture/logs",
    skills_dir="/notebooks/cogmem_new_architecture/skills",
    active_model_hf=NOTEBOOK_CONFIG.model_name,
    base_model=NOTEBOOK_CONFIG.model_name,
    quantization_bits=8,
    use_dora=False,
    generator_rank=8,
    generator_alpha=16,
    verifier_rank=8,
    verifier_alpha=16,
    generator_batch_size=1,
    generator_max_seq_length=2048,
    generator_sft_epochs=3,
    generator_dpo_epochs=2,
    verifier_epochs=2,
    min_dpo_pairs=24,
    skill_min_distinct_tasks=2,
    skill_curriculum_examples_per_card=0,
    skill_retrieval_min_score=4.5,
    skill_retrieval_strict_min_score=6.0,
    skill_retrieval_min_promoted_families_for_broad_match=3,
    skill_runtime_disable_min_retrieved=8,
    skill_runtime_disable_min_hurt=3,
    skill_runtime_disable_hurt_rate=0.10,
    episode_summary_max_code_lines=8,
    episode_retrieval_allow_first_attempt=False,
    episode_retrieval_min_score=7.0,
    episode_retrieval_retry_min_score=5.5,
    episode_runtime_disable_min_retrieved=8,
    episode_runtime_disable_min_hurt=3,
    episode_runtime_disable_hurt_rate=0.10,
    skill_route_promotion_min_tasks=2,
    skill_route_promotion_min_delta_passed=1,
    skill_route_promotion_max_regression_rate=0.10,
    min_promoted_skills_for_adapter=3,
    min_skill_families_for_adapter=2,
    min_skill_training_pairs_for_adapter=48,
    skill_route_gate_task_limit=30,
    skill_route_gate_min_delta_passed=1,
    skill_route_gate_max_regression_rate=0.10,
    allowed_manifest_ids=[manifest["manifest_id"]],
    require_manifest_match=True,
    bigcodebench_eval_label="bigcodebench_cl",
)

print("Manifest:", manifest["manifest_id"])
print("Train tasks:", len(train_tasks))
print("Dev tasks  :", len(dev_tasks))
print("Test tasks :", len(test_tasks))
print("Memory bank:", NOTEBOOK_CONFIG.memory_bank_path)
print("Skill cards:", NOTEBOOK_CONFIG.skills_path)


In [ ]:
# Cell 4: Load local HF runtime for episode collection
import torch

base_model, tokenizer, llm_client = load_new_arch_runtime(
    model_name=NOTEBOOK_CONFIG.model_name,
)

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Runtime loaded. Free VRAM: {free:.1f} GB")


In [ ]:
# Cell 5: Collect typed episodes on the train split
collection_result = run_new_arch_episode_collection(
    train_tasks,
    llm_client,
    config=NOTEBOOK_CONFIG,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    reset_progress=RESET_COLLECTION_PROGRESS,
    verbose=True,
)

print()
print("=" * 60)
print("NEW-ARCH EPISODE COLLECTION COMPLETE")
print("Tasks processed    :", collection_result["tasks_processed"])
print("New episodes       :", collection_result["new_episodes"])
print("Episodes total     :", collection_result["episodes_total"])
print("New successes      :", collection_result["successes_this_run"])
print("Elapsed (min)      :", round(collection_result["elapsed_minutes"], 1))
print("Progress file      :", collection_result["progress_path"])
print("Memory bank path   :", collection_result["memory_bank_path"])
print("Summary metrics    :", collection_result["summary_metrics"])


In [ ]:
# Cell 6: Inspect typed episodic memory bank
from collections import Counter
from cogmem.memory.memory_bank import MemoryBank

bank = MemoryBank.load(NOTEBOOK_CONFIG.memory_bank_path)
metrics = bank.summary_metrics()
print("Episodes:", len(bank))
print("Summary metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

task_type_counts = Counter(ep.get("task_type", "general") for ep in bank)
error_family_counts = Counter(ep.get("error_family") or "None" for ep in bank)
split_counts = Counter(ep.get("split_name") or "unspecified" for ep in bank)

print()
print("Task types:")
for key, value in sorted(task_type_counts.items()):
    print(f"  {key}: {value}")

print()
print("Error families:")
for key, value in error_family_counts.most_common(10):
    print(f"  {key}: {value}")

print()
print("Split counts:")
for key, value in sorted(split_counts.items()):
    print(f"  {key}: {value}")


In [ ]:
# Cell 7: Build + validate procedural skill cards
skill_result = build_new_arch_skill_cards(
    NOTEBOOK_CONFIG.memory_bank_path,
    COGMEM_CONFIG,
    skills_path=NOTEBOOK_CONFIG.skills_path,
)

print("Episodes total     :", skill_result["episodes_total"])
print("Eligible episodes  :", skill_result["eligible_episodes"])
print("Available episodes :", skill_result["available_episodes"])
print("Holdout episodes   :", skill_result["holdout_episodes"])
print("Task type counts   :", skill_result["task_type_counts"])
print("Skill summary      :", skill_result["skill_summary"])
print("Training pairs     :", skill_result["training_pairs"])
print("Preference pairs   :", skill_result["preference_pairs"])

print()
print("Top skill cards:")
for row in skill_result["skill_rows"]:
    print("Skill:", row["skill_id"])
    print(
        "  status:", row["status"],
        "| task_type:", row["task_type"],
        "| domain:", row["domain"],
        "| error_family:", row["error_family"],
    )
    print(
        "  evidence:", row["source_episode_count"],
        "| distinct_tasks:", row["distinct_task_count"],
        "| matched:", row["matched_episodes"],
        "| matched_tasks:", row["matched_tasks"],
        "| confidence:", round(row["confidence"], 3),
        "| transfer:", round(row["transfer_gain"], 3),
        "| negative_transfer:", round(row["negative_transfer_rate"], 3),
        "| success_rate:", round(row["success_rate"], 3),
        "| route_test:", row.get("route_test_status"),
    )
    print(
        "  route_delta:", row.get("route_delta_passed"),
        "| route_regression:", row.get("route_regression_rate"),
    )
    print("  triggers:", row["triggers"])
    print("  activation_conditions:", row["activation_conditions"])
    print("  plan_steps:", row["plan_steps"])
    print("  stop_conditions:", row["stop_conditions"])
    print("  anti_patterns:", row["anti_patterns"])


In [ ]:
# Cell 8: Optional Q-STaR consolidation cycle (adapter training)
if RUN_QSTAR_TRAINING:
    import gc
    import torch

    base_model = None
    tokenizer = None
    llm_client = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    qstar_result = run_new_arch_qstar_cycle(
        NOTEBOOK_CONFIG.memory_bank_path,
        COGMEM_CONFIG,
        cycle=0,
        skill_cards_path=skill_result["skills_path"],
        eval_tasks=dev_tasks,
        eval_task_limit=COGMEM_CONFIG.skill_route_gate_task_limit,
    )
    print(json.dumps(qstar_result, indent=2, ensure_ascii=False))
else:
    print("RUN_QSTAR_TRAINING is False. Set it to True to train generator/verifier adapters.")
    print("Promoted skills available:", len(skill_result["promoted_skill_ids"]))
    print("Skill cards path:", skill_result["skills_path"])


In [ ]:
# Cell 9: Evaluate base, retrieved-skill, retrieved-episode, router, and adapter routes
EVAL_SPLIT = "dev"  # or "test"
EVAL_TASK_LIMIT = 30  # set to None for the full split
EVAL_MAX_ATTEMPTS = 2
EVAL_SKILL_TOP_K = COGMEM_CONFIG.skill_retrieval_top_k

eval_tasks = dev_tasks if EVAL_SPLIT == "dev" else test_tasks
skill_cards_path = skill_result.get("skills_path") if "skill_result" in globals() else NOTEBOOK_CONFIG.skills_path
adapter_path = qstar_result.get("generator_path") if "qstar_result" in globals() else None

import json

eval_result = compare_new_arch_routes(
    eval_tasks,
    model_name=NOTEBOOK_CONFIG.model_name,
    skill_cards_path=skill_cards_path,
    episode_memory_path=NOTEBOOK_CONFIG.memory_bank_path,
    adapter_path=adapter_path,
    skill_top_k=EVAL_SKILL_TOP_K,
    config=COGMEM_CONFIG,
    task_limit=EVAL_TASK_LIMIT,
    max_tokens=NOTEBOOK_CONFIG.max_tokens,
    temperature=0.0,
    max_attempts=EVAL_MAX_ATTEMPTS,
    eval_timeout=NOTEBOOK_CONFIG.eval_timeout,
    verbose=True,
)

persist_result = persist_route_memory_utility(
    skill_cards_path,
    NOTEBOOK_CONFIG.memory_bank_path,
    eval_result.get("comparisons", {}),
)

summary = {
    "split": EVAL_SPLIT,
    "task_count": eval_result["task_count"],
    "base_passed": eval_result["base"]["passed"],
    "base_pass_rate": eval_result["base"]["pass_rate"],
    "base_plus_skill_passed": eval_result.get("base_plus_skill", {}).get("passed"),
    "base_plus_skill_pass_rate": eval_result.get("base_plus_skill", {}).get("pass_rate"),
    "base_plus_skill_delta_passed": eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("delta_passed"),
    "base_plus_skill_regression_rate": eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("regression_rate"),
    "base_plus_episode_passed": eval_result.get("base_plus_episode", {}).get("passed"),
    "base_plus_episode_pass_rate": eval_result.get("base_plus_episode", {}).get("pass_rate"),
    "base_plus_episode_delta_passed": eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("delta_passed"),
    "base_plus_episode_regression_rate": eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("regression_rate"),
    "base_plus_router_passed": eval_result.get("base_plus_router", {}).get("passed"),
    "base_plus_router_pass_rate": eval_result.get("base_plus_router", {}).get("pass_rate"),
    "base_plus_router_delta_passed": eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("delta_passed"),
    "base_plus_router_regression_rate": eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("regression_rate"),
    "adapter_passed": eval_result.get("adapter", {}).get("passed"),
    "adapter_pass_rate": eval_result.get("adapter", {}).get("pass_rate"),
    "adapter_plus_router_passed": eval_result.get("adapter_plus_router", {}).get("passed"),
    "adapter_plus_router_pass_rate": eval_result.get("adapter_plus_router", {}).get("pass_rate"),
    "delta_passed": eval_result.get("delta_passed"),
    "delta_pass_rate": eval_result.get("delta_pass_rate"),
    "improved_task_ids": eval_result.get("improved_task_ids", [])[:10],
    "regressed_task_ids": eval_result.get("regressed_task_ids", [])[:10],
    "base_plus_skill_selected": eval_result.get("base_plus_skill", {}).get("selected_skill_ids", {}),
    "base_plus_episode_selected": eval_result.get("base_plus_episode", {}).get("selected_episode_ids", {}),
    "base_plus_router_routes": eval_result.get("base_plus_router", {}).get("selected_route_counts", {}),
    "adapter_plus_router_routes": eval_result.get("adapter_plus_router", {}).get("selected_route_counts", {}),
    "utility_persisted": persist_result,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Base+skill utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
print("Base+episode utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))
print("Base+router skill utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
print("Base+router episode utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))
if adapter_path:
    print("Adapter+router skill utility:")
    print(json.dumps(eval_result.get("comparisons", {}).get("adapter_plus_router_vs_adapter", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
    print("Adapter+router episode utility:")
    print(json.dumps(eval_result.get("comparisons", {}).get("adapter_plus_router_vs_adapter", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))
